# Fetch Reference Genomes / 抓取参考基因组

**Purpose / 目的**

Download the three reference genomes that anchor the active-learning dry-lab pipeline:
1. *Xanthomonas campestris* pv. *campestris* ATCC 33913 (Xcc) — host receptor source.
2. *Xanthomonas* phage phiL7 — RBP scaffold (already present in `00_raw_data/phage/EU717894.1/`).
3. T7 phage — gp17 RBP positive control for ELISA optimization.

下载支撑主动学习 dry-lab pipeline 的三个参考基因组：
1. *Xanthomonas campestris* pv. *campestris* ATCC 33913 (Xcc) —— 宿主受体序列来源。
2. *Xanthomonas* 噬菌体 phiL7 —— RBP scaffold（已经在 `00_raw_data/phage/EU717894.1/` 里）。
3. T7 噬菌体 —— ELISA 优化的 gp17 RBP positive control。

**Convention / 编写约定**

This notebook follows the project's bilingual development convention. Once it runs end-to-end and outputs match the verification cell, we will freeze it as `fetch_reference_genomes.py` and import it from other notebooks.

本 notebook 遵循专案的双语开发约定。等它能端到端跑通并通过验证 cell 后，会冻结为 `fetch_reference_genomes.py`，让其他 notebook import 使用。

In [ ]:
# Imports + config / 引入与配置
import subprocess
import shutil
import zipfile
from pathlib import Path
import pandas as pd

# Anchor paths to repo root, never hard-code absolute paths
# 路径锚定到 repo root，绝对不要写死绝对路径
REPO_ROOT = Path.cwd().resolve().parents[1]
RAW_DATA = REPO_ROOT / "00_raw_data"
PHAGE_DIR = RAW_DATA / "phage"
BACTERIA_DIR = RAW_DATA / "bacteria"

print(f"Repo root / 仓库根目录: {REPO_ROOT}")
print(f"Phage dir / 噬菌体目录: {PHAGE_DIR}  exists={PHAGE_DIR.exists()}")
print(f"Bacteria dir / 细菌目录: {BACTERIA_DIR}  exists={BACTERIA_DIR.exists()}")

## Reference genome registry / 参考基因组登记

Each entry below is a `(category, assembly_accession, label, expected_dir_name)` tuple. We use the RefSeq assembly accession (`GCF_*`) for `datasets` CLI download because it pulls a clean bundle (genome.fna + cds.fna + protein.faa).

下面每条 entry 是 `(类别, assembly accession, 标签, 预期目录名)` 四元组。我们用 RefSeq assembly accession (`GCF_*`) 调用 `datasets` CLI 下载，因为它会打包一个完整的目录（genome.fna + cds.fna + protein.faa）。

In [ ]:
# Registry of reference genomes / 参考基因组清单
REFERENCES = [
    # (category, assembly_acc, nucleotide_acc, label, target_subdir_name)
    ("bacteria", "GCF_000007145.1", "AE008922", "Xcc ATCC 33913", "AE008922"),
    ("phage",    "GCF_000840885.1", "NC_001604", "T7 phage",       "NC_001604"),
    # phiL7 is already in 00_raw_data/phage/EU717894.1/ — included here for verification only
    # phiL7 已经在 00_raw_data/phage/EU717894.1/ 里 —— 这里列出仅用于完整性校验
    ("phage",    None,              "EU717894", "Xanthomonas phage phiL7", "EU717894.1"),
]

df_ref = pd.DataFrame(REFERENCES, columns=["category", "assembly_acc", "nucleotide_acc", "label", "subdir"])
df_ref

## Fetch helpers / 抓取辅助函数

Two strategies depending on what's available on NCBI:
- **`datasets` CLI** for full RefSeq assemblies (preferred — gets genome + cds + protein in one shot).
- **Biopython Entrez fallback** for nucleotide-only records (when no assembly is available).

根据 NCBI 上有什么，分两种策略：
- **`datasets` CLI**：处理完整的 RefSeq assembly（首选 —— 一次拿 genome + cds + protein）。
- **Biopython Entrez fallback**：处理只有 nucleotide record 的情况。

In [ ]:
def fetch_assembly_via_datasets(assembly_acc: str, target_dir: Path) -> bool:
    """Download a RefSeq assembly via NCBI datasets CLI.
    通过 NCBI datasets CLI 下载 RefSeq assembly。

    Returns True on success, False otherwise.
    成功返回 True，失败返回 False。
    """
    target_dir.mkdir(parents=True, exist_ok=True)

    # Download to a temporary zip in the target directory
    # 下载到 target 目录里的一个临时 zip
    zip_path = target_dir / "dataset.zip"
    cmd = [
        "datasets", "download", "genome", "accession", assembly_acc,
        "--include", "genome,cds,protein",
        "--filename", str(zip_path),
    ]
    print(f"  Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  [ERROR] datasets CLI failed: {result.stderr.strip()}")
        return False

    # Unzip the bundle and flatten the layout
    # 解压并把布局拍平
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(target_dir / "_extracted")

    # NCBI bundle path: ncbi_dataset/data/<accession>/{genomic.fna, cds_from_genomic.fna, protein.faa}
    extracted_root = target_dir / "_extracted" / "ncbi_dataset" / "data" / assembly_acc
    if not extracted_root.exists():
        print(f"  [ERROR] Expected {extracted_root} not found after unzip")
        return False

    # Map NCBI's filenames to our convention
    # 把 NCBI 的文件名映射到我们的命名约定
    rename_map = {
        "genomic.fna": "genome.fna",
        "cds_from_genomic.fna": "cds.fna",
        "protein.faa": "proteins.faa",
    }
    for src_name, dst_name in rename_map.items():
        src = extracted_root / src_name
        if src.exists():
            shutil.move(str(src), str(target_dir / dst_name))
        else:
            print(f"  [WARN] {src_name} not in bundle")

    # Cleanup / 清理
    shutil.rmtree(target_dir / "_extracted", ignore_errors=True)
    zip_path.unlink(missing_ok=True)

    print(f"  [OK] {assembly_acc} → {target_dir}")
    return True

In [ ]:
def verify_genome_dir(target_dir: Path) -> dict:
    """Quick sanity check on a downloaded genome directory.
    对下载好的基因组目录做一次快速 sanity check。
    """
    expected = ["genome.fna", "proteins.faa"]  # cds.fna optional
    status = {"dir": str(target_dir), "exists": target_dir.exists()}
    for fname in expected:
        fpath = target_dir / fname
        status[fname] = fpath.exists()
        if fpath.exists():
            status[f"{fname}_size_kb"] = round(fpath.stat().st_size / 1024, 1)
    return status

## Run fetches / 执行抓取

Idempotent — skips entries whose target directory already contains a `genome.fna`. To force re-download, delete the target directory first.

幂等 —— 如果 target 目录已经有 `genome.fna` 就跳过。要强制重新下载，先删掉 target 目录。

In [ ]:
results = []
for cat, asm_acc, nuc_acc, label, subdir in REFERENCES:
    base = PHAGE_DIR if cat == "phage" else BACTERIA_DIR
    target = base / subdir

    print(f"\n=== {label}  ({nuc_acc}) ===")
    if (target / "genome.fna").exists():
        print(f"  [SKIP] Already exists at {target}")
        results.append((label, nuc_acc, "skipped", str(target)))
        continue

    if asm_acc is None:
        print(f"  [SKIP] No assembly accession — manual fetch required for {nuc_acc}")
        results.append((label, nuc_acc, "manual_required", str(target)))
        continue

    success = fetch_assembly_via_datasets(asm_acc, target)
    results.append((label, nuc_acc, "ok" if success else "failed", str(target)))

df_results = pd.DataFrame(results, columns=["label", "accession", "status", "path"])
df_results

## Verification / 验证

Confirm every reference has the expected files and reasonable file sizes.

确认每个参考基因组都有预期的文件以及合理的档案大小。

In [ ]:
verification = []
for cat, asm_acc, nuc_acc, label, subdir in REFERENCES:
    base = PHAGE_DIR if cat == "phage" else BACTERIA_DIR
    status = verify_genome_dir(base / subdir)
    status["label"] = label
    verification.append(status)

df_verify = pd.DataFrame(verification)
df_verify

## Next steps / 下一步

1. Hand off `00_raw_data/{phage,bacteria}/<accession>/proteins.faa` to **Module 02 annotation** (PHANOTATE on phage, Prodigal on bacteria) and **Module 03 RBP identification** (PhageRBPdetect on phage `proteins.faa`).
2. Once this notebook runs end-to-end and the verification table shows all-green, freeze it as `fetch_reference_genomes.py` per project convention.

1. 把 `00_raw_data/{phage,bacteria}/<accession>/proteins.faa` 传给 **模块 02 注释**（PHANOTATE 跑噬菌体、Prodigal 跑细菌）跟 **模块 03 RBP 识别**（PhageRBPdetect 跑噬菌体的 `proteins.faa`）。
2. 等这个 notebook 端到端跑通，验证表全绿后，按专案约定冻结成 `fetch_reference_genomes.py`。